# 01. Exploración del conjunto de imágenes — Banana vs. Guava

**Proyecto:** Clasificador de guayaba y banana.  
**Objetivo:** caracterizar las imágenes antes del preprocesamiento y entrenamiento.

El cuaderno localiza imágenes, identifica las clases a partir de carpetas `Banana` y `Guava`, revisa legibilidad, dimensiones, formatos, modos de color, tamaños y duplicados exactos. Genera gráficas, una galería de muestras y archivos CSV. No modifica las imágenes originales.

Ejecuta las celdas en orden.

## 1. Configuración

In [ ]:
from pathlib import Path
import hashlib, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)

RAIZ_PROYECTO = Path.cwd().resolve()
if RAIZ_PROYECTO.name.lower() == "notebooks":
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent

CARPETA_RESULTADOS = RAIZ_PROYECTO / "results"
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)
EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
EXCLUIR = {".git", ".venv", ".venv314", "__pycache__", ".ipynb_checkpoints", "site-packages"}

print("Proyecto:", RAIZ_PROYECTO)
print("Resultados:", CARPETA_RESULTADOS)

## 2. Localización de imágenes

Se buscan imágenes recursivamente, excluyendo entornos virtuales y Git. La etiqueta se toma de los nombres de las carpetas que contengan `Banana`, `Guava` o `Guayaba`.

In [ ]:
def esta_excluida(ruta):
    return any(parte.lower() in EXCLUIR for parte in ruta.parts)

rutas = [p for p in RAIZ_PROYECTO.rglob("*")
         if p.is_file() and p.suffix.lower() in EXTENSIONES and not esta_excluida(p)]

def clase_de(ruta):
    partes = [p.lower() for p in ruta.parts]
    if any("banana" in p for p in partes): return "Banana"
    if any("guava" in p or "guayaba" in p for p in partes): return "Guava"
    return "Sin_clase"

print("Imágenes encontradas:", len(rutas))
display(pd.Series([clase_de(p) for p in rutas]).value_counts().to_frame("archivos"))
if not rutas:
    raise FileNotFoundError("No se encontraron imágenes. Revisa la ubicación del dataset y las carpetas Banana/Guava.")

## 3. Auditoría técnica

Se verifica que cada imagen pueda abrirse y se registran sus propiedades. El hash SHA-256 permite detectar archivos idénticos.

In [ ]:
registros = []
for ruta in rutas:
    r = {"ruta": str(ruta.relative_to(RAIZ_PROYECTO)), "archivo": ruta.name,
         "clase": clase_de(ruta), "extension": ruta.suffix.lower(),
         "bytes": ruta.stat().st_size, "ancho": None, "alto": None,
         "modo_color": None, "formato_real": None, "estado": "no_legible", "sha256": None}
    try:
        with Image.open(ruta) as im: im.verify()
        with Image.open(ruta) as im:
            r.update(ancho=im.width, alto=im.height, modo_color=im.mode,
                     formato_real=im.format, estado="valida")
        h = hashlib.sha256()
        with open(ruta, "rb") as f:
            for bloque in iter(lambda: f.read(1024*1024), b""): h.update(bloque)
        r["sha256"] = h.hexdigest()
    except (UnidentifiedImageError, OSError, ValueError):
        pass
    registros.append(r)

df = pd.DataFrame(registros)
validas = df[df.estado == "valida"].copy()
print("Total:", len(df), "| Válidas:", len(validas), "| No legibles:", (df.estado != "valida").sum())
print("Sin etiqueta:", (df.clase == "Sin_clase").sum())
display(df.head())

## 4. Distribución de clases

In [ ]:
resumen = (validas.groupby("clase").size().rename("cantidad").to_frame())
resumen["porcentaje"] = 100 * resumen["cantidad"] / max(len(validas), 1)
display(resumen.round(2))

conteos = validas.clase.value_counts()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(conteos.index, conteos.values)
ax[0].set(title="Imágenes por clase", ylabel="Cantidad")
ax[0].grid(axis="y", alpha=.25)
if len(conteos):
    ax[1].pie(conteos.values, labels=conteos.index, autopct="%1.1f%%", startangle=90)
    ax[1].set_title("Proporción por clase")
plt.tight_layout()
plt.show()

## 5. Dimensiones, razón de aspecto, formatos y color

In [ ]:
validas["pixeles"] = validas.ancho * validas.alto
validas["razon_aspecto"] = validas.ancho / validas.alto
validas["tamano_mb"] = validas.bytes / (1024**2)

print("Dimensiones más frecuentes:")
display(validas.groupby(["ancho","alto"]).size().sort_values(ascending=False).head(15).to_frame("cantidad"))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(validas.ancho.dropna(), bins=20); ax[0].set_title("Distribución del ancho"); ax[0].set_xlabel("Píxeles")
ax[1].hist(validas.alto.dropna(), bins=20); ax[1].set_title("Distribución del alto"); ax[1].set_xlabel("Píxeles")
ax[2].hist(validas.razon_aspecto.dropna(), bins=20); ax[2].set_title("Razón de aspecto"); ax[2].set_xlabel("Ancho / alto")
for a in ax: a.set_ylabel("Frecuencia"); a.grid(alpha=.2)
plt.tight_layout(); plt.show()

print("Formatos:"); display(validas.formato_real.value_counts().to_frame("cantidad"))
print("Modos de color:"); display(validas.modo_color.value_counts().to_frame("cantidad"))

## 6. Tamaño de archivo y duplicados exactos

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
for clase, grupo in validas.groupby("clase"):
    ax.hist(grupo.tamano_mb, bins=25, alpha=.55, label=clase)
ax.set(title="Distribución del tamaño de archivo", xlabel="MB", ylabel="Frecuencia")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

duplicados = validas[validas.sha256.duplicated(keep=False)].sort_values("sha256")
print("Archivos en grupos de duplicados exactos:", len(duplicados))
display(duplicados[["ruta","clase","sha256"]].head(30)) if len(duplicados) else print("No se detectaron duplicados exactos.")

## 7. Galería de muestras

In [ ]:
def mostrar_muestras(datos, clase, n=8):
    grupo = datos[datos.clase == clase]
    if grupo.empty:
        print("Sin imágenes para", clase); return
    muestra = grupo.sample(n=min(n, len(grupo)), random_state=42)
    cols = 4; filas = int(np.ceil(len(muestra)/cols))
    fig, ejes = plt.subplots(filas, cols, figsize=(14, 3.5*filas))
    ejes = np.array(ejes).reshape(-1)
    for ax in ejes: ax.axis("off")
    for ax, (_, fila) in zip(ejes, muestra.iterrows()):
        with Image.open(RAIZ_PROYECTO / fila.ruta) as im: ax.imshow(im.convert("RGB"))
        ax.set_title(f"{fila.archivo[:26]}\n{fila.ancho}×{fila.alto}")
        ax.axis("off")
    fig.suptitle(f"Muestras aleatorias: {clase}")
    plt.tight_layout(); plt.show()

for c in ["Banana", "Guava"]:
    mostrar_muestras(validas, c)

## 8. Exportación de resultados

In [ ]:
df.to_csv(CARPETA_RESULTADOS / "exploracion_imagenes.csv", index=False, encoding="utf-8-sig")
resumen.to_csv(CARPETA_RESULTADOS / "resumen_clases.csv", encoding="utf-8-sig")
duplicados.to_csv(CARPETA_RESULTADOS / "duplicados_exactos.csv", index=False, encoding="utf-8-sig")
print("Resultados guardados en:", CARPETA_RESULTADOS)

## 9. Interpretación científica

Después de ejecutar el cuaderno, redacta conclusiones basadas en los resultados reales:

1. ¿Cuántas imágenes válidas hay por clase y qué porcentaje representa cada una?
2. ¿Existe desbalance entre Banana y Guava? ¿Cómo puede afectar la evaluación?
3. ¿Las dimensiones son uniformes?
4. ¿Qué formatos y modos de color predominan?
5. ¿Se detectaron duplicados exactos?
6. ¿Qué diferencias visuales hay en iluminación, fondo, orientación, distancia, madurez o defectos?

**Nota:** esta primera etapa tiene como objetivo la clasificación binaria por fruta. Las etiquetas de calidad (`Class_A`, `Class_B`, `Defect`) se reservarán para un experimento independiente. No declares que el dataset está balanceado o libre de problemas sin revisar las salidas del cuaderno.